# Data Clean Rooms

**Privacy-preserving collaboration framework.** Multiple parties analyse combined data without exposing raw data to each other. Only approved analysis templates return aggregate results.

| Concept | Description |
|---------|-------------|
| **Provider** | Creates the collaboration and defines approved analysis templates |
| **Consumer** | Joins the collaboration, registers data, and runs approved analyses |
| **Template** | Pre-approved query pattern (e.g., overlap, attribution) — enforces what can be computed |
| **Differential Privacy** | Optional noise injection to protect individual-level information in results |

## How Clean Rooms Work

```
┌──────────────┐         ┌──────────────┐
│   Provider    │         │   Consumer   │
│  (Party A)    │         │  (Party B)   │
│               │         │              │
│  Raw Data A   │◄───────►│  Raw Data B  │
│  (private)    │  Clean  │  (private)   │
│               │  Room   │              │
│               │         │              │
│       ┌───────┴─────────┴───────┐      │
│       │  Approved Templates     │      │
│       │  (overlap, attribution) │      │
│       │                         │      │
│       │  ► Aggregate Results ◄  │      │
│       └─────────────────────────┘      │
└──────────────┘         └──────────────┘
```

Only aggregate/approved results are returned. Neither party sees the other's raw data.

## 1. Simulated Setup — Advertiser & Publisher Data

Since DCR is managed via the Snowflake Clean Room API, we simulate the concept with secure views and aggregation.

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE clean_room_demo;
CREATE SCHEMA clean_room_demo.advertiser;
CREATE SCHEMA clean_room_demo.publisher;
CREATE SCHEMA clean_room_demo.clean_room;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS clean_room_admin;
GRANT ROLE clean_room_admin TO ROLE SYSADMIN;
GRANT USAGE ON DATABASE clean_room_demo TO ROLE clean_room_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE clean_room_demo TO ROLE clean_room_admin;
GRANT CREATE VIEW ON SCHEMA clean_room_demo.clean_room TO ROLE clean_room_admin;

-- Advertiser data (Party A)
USE ROLE ACCOUNTADMIN;
CREATE TABLE clean_room_demo.advertiser.customers AS
SELECT c_custkey AS customer_id,
       SHA2(c_name) AS hashed_email,
       c_mktsegment AS segment,
       c_nationkey AS nation_id
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
LIMIT 5000;

-- Publisher data (Party B)
CREATE TABLE clean_room_demo.publisher.subscribers AS
SELECT c_custkey + 2500 AS subscriber_id,
       SHA2(c_name) AS hashed_email,
       CASE MOD(c_custkey, 4)
           WHEN 0 THEN 'news'
           WHEN 1 THEN 'sports'
           WHEN 2 THEN 'tech'
           ELSE 'entertainment'
       END AS content_preference,
       MOD(c_custkey, 30) + 1 AS page_views
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
LIMIT 5000;

GRANT SELECT ON ALL TABLES IN SCHEMA clean_room_demo.advertiser TO ROLE clean_room_admin;
GRANT SELECT ON ALL TABLES IN SCHEMA clean_room_demo.publisher TO ROLE clean_room_admin;

## 2. Clean Room Template — Overlap Analysis

This simulates an approved clean room template that returns only aggregate overlap statistics.

In [ ]:
USE ROLE clean_room_admin;

CREATE OR REPLACE SECURE VIEW clean_room_demo.clean_room.v_overlap_analysis AS
SELECT
    a.segment AS advertiser_segment,
    p.content_preference AS publisher_content,
    COUNT(*) AS overlap_count,
    AVG(p.page_views) AS avg_page_views
FROM clean_room_demo.advertiser.customers a
INNER JOIN clean_room_demo.publisher.subscribers p
    ON a.hashed_email = p.hashed_email
GROUP BY a.segment, p.content_preference
HAVING COUNT(*) >= 5;

## 3. Run Overlap Analysis (Aggregate Only)

The consumer can only see aggregate results — no individual-level data.

In [ ]:
SELECT * FROM clean_room_demo.clean_room.v_overlap_analysis
ORDER BY overlap_count DESC;

## 4. Clean Room Template — Audience Sizing

Another approved template: count of matching users by segment, without exposing individual records.

In [ ]:
USE ROLE clean_room_admin;

CREATE OR REPLACE SECURE VIEW clean_room_demo.clean_room.v_audience_sizing AS
SELECT
    a.segment,
    COUNT(DISTINCT a.customer_id) AS total_advertiser_customers,
    COUNT(DISTINCT p.subscriber_id) AS matched_subscribers,
    ROUND(COUNT(DISTINCT p.subscriber_id) * 100.0 / NULLIF(COUNT(DISTINCT a.customer_id), 0), 2) AS match_rate_pct
FROM clean_room_demo.advertiser.customers a
LEFT JOIN clean_room_demo.publisher.subscribers p
    ON a.hashed_email = p.hashed_email
GROUP BY a.segment;

SELECT * FROM clean_room_demo.clean_room.v_audience_sizing;

## 5. Using Snowflake Data Clean Rooms (Production)

In production, Snowflake Data Clean Rooms are managed via:

1. **Snowflake Clean Room API** — Provider creates collaborations and defines templates
2. **Consumer joins** the collaboration and runs approved analyses
3. **Differential privacy** can be applied to results
4. **Template enforcement** ensures only approved queries execute

Key steps:
- Provider: Create collaboration → Define templates → Invite consumer
- Consumer: Accept invitation → Register data → Run analysis
- Results: Aggregate only, with optional differential privacy

> **Note:** DCR setup requires the Snowflake Data Clean Rooms product. Contact your Snowflake representative for access.

## 6. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP DATABASE IF EXISTS clean_room_demo;
DROP ROLE IF EXISTS clean_room_admin;

## References

- [Snowflake Data Clean Rooms Overview](https://docs.snowflake.com/en/user-guide/cleanrooms/introduction)
- [Clean Room Collaboration API](https://docs.snowflake.com/en/sql-reference/sql/create-clean-room)
- [Privacy-Preserving Data Sharing](https://docs.snowflake.com/en/user-guide/cleanrooms/privacy)
- [Clean Room Templates](https://docs.snowflake.com/en/user-guide/cleanrooms/templates)
- [Differential Privacy in Clean Rooms](https://docs.snowflake.com/en/user-guide/cleanrooms/differential-privacy)